In [7]:

import sys, os, json, time, traceback
sys.path.insert(0, os.path.abspath('..'))

import importlib
import httpx
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

import data_pipeline.config_variables as _cv
importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import data_pipeline.config_cluster as _cc
importlib.reload(_cc)
from data_pipeline.config_cluster import (
    N_CLUSTERS_LOCAL, TEST_MODE, TEST_LA_CODES, WAVE, HIERARCHICAL_CLUSTER
)

import data_pipeline.helpers.cluster_summary as _cs
importlib.reload(_cs)

import data_pipeline.helpers.llm_prompts as _lp
importlib.reload(_lp)
from data_pipeline.helpers.llm_prompts import (
    CODEBOOK_STR, SYSTEM_PROMPT, sanitise_name, build_user_prompt, resolve_emp_label
)

# ── Config ────────────────────────────────────────────────────────────────────
MAX_PIDPS_PER_GROUP   = 100   # top N PIDPs (by weight) per LA × group
MODEL                 = "gemini-2.5-flash"
RETRY_DELAY           = 10
_GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

hier_col = f"{WAVE}_{HIERARCHICAL_CLUSTER}" if HIERARCHICAL_CLUSTER else None

print(f"WAVE={WAVE}  HIERARCHICAL_CLUSTER={HIERARCHICAL_CLUSTER}  MODEL={MODEL}")
print(f"MAX_PIDPS_PER_GROUP={MAX_PIDPS_PER_GROUP}")

# ── Profile encoding (compact numeric codes instead of NL prose) ──────────────
# Format: age|sex|eth|edu|emp|mar|ten|hh|health  (? = unknown/missing)
_PROFILE_COLS = [
    (f"{WAVE}_doby_dv_eng", "age"),
    (f"{WAVE}_sex_dv",      "sex"),
    (f"{WAVE}_racel_dv",    "eth"),
    (f"{WAVE}_hiqual_dv",   "edu"),
    (f"{WAVE}_jbstat",      "emp"),
    (f"{WAVE}_marstat_dv",  "mar"),
    (f"{WAVE}_tenure_dv",   "ten"),
    (f"{WAVE}_hhtype_dv",   "hh"),
    (f"{WAVE}_scsf1",       "health"),
]

def _encode_profile(row, avail_cols: list) -> str:
    parts = []
    for col, _ in _PROFILE_COLS:
        if col not in avail_cols:
            parts.append("?")
            continue
        v = row[col]
        try:
            fv = float(v)
            parts.append("?" if (pd.isna(fv) or fv < 0) else str(int(fv)))
        except (TypeError, ValueError):
            parts.append("?")
    return "|".join(parts)

# ── Paths ─────────────────────────────────────────────────────────────────────
NL_PROFILE = Path(f"../{DATA_FOLDER}/5_add_nl_strings/{WAVE}_with_nl_profile.pkl")
LA_COUNTS  = Path(f"../{DATA_FOLDER}/10_synthetic_population/pidp_la_counts.csv")
API_OUT    = Path("../api/data/clusters/local_llm_gemini_clusters.csv")

for p in (NL_PROFILE, LA_COUNTS):
    if not p.exists():
        raise FileNotFoundError(f"{p} — check pipeline prerequisites.")

API_OUT.parent.mkdir(parents=True, exist_ok=True)

# ── API key ───────────────────────────────────────────────────────────────────
load_dotenv(Path("..") / ".env")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "").strip()
if not GOOGLE_API_KEY:
    raise EnvironmentError("GOOGLE_API_KEY not set. Add to .env or export in shell.")
print(f"GOOGLE_API_KEY loaded (length={len(GOOGLE_API_KEY)}, prefix={GOOGLE_API_KEY[:6]}…)", flush=True)

# ── Native Gemini caller — x-goog-api-key header, thinking enabled ────────────
def _gemini_chat(system: str, user: str, temperature: float = 0.3, timeout: int = 300) -> str:
    """Call Gemini native API. Prints thinking, returns final response text."""
    payload = {
        "systemInstruction": {"parts": [{"text": system}]},
        "contents": [{"role": "user", "parts": [{"text": user}]}],
        "generationConfig": {
            "responseMimeType": "application/json",
            "temperature": temperature,
            "thinkingConfig": {"includeThoughts": True},
        },
    }
    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": GOOGLE_API_KEY,
    }
    print(f"  POST {_GEMINI_URL}", flush=True)
    with httpx.Client(timeout=timeout) as http:
        resp = http.post(_GEMINI_URL, headers=headers, json=payload)
    print(f"  HTTP {resp.status_code}", flush=True)
    if resp.status_code != 200:
        print(f"  Response body: {resp.text}", flush=True)
        resp.raise_for_status()
    data = resp.json()
    parts = data["candidates"][0]["content"]["parts"]
    thoughts = [p["text"] for p in parts if p.get("thought")]
    response = [p["text"] for p in parts if not p.get("thought")]
    if thoughts:
        print(f"\n── Gemini thinking ──\n{''.join(thoughts)}\n────────────────────\n", flush=True)
    return "".join(response)

print("Gemini client ready (native API, x-goog-api-key, thinking enabled).")


WAVE=k  HIERARCHICAL_CLUSTER=jbstat_eng  MODEL=gemini-2.5-flash
MAX_PIDPS_PER_GROUP=100
GOOGLE_API_KEY loaded (length=39, prefix=AIzaSy…)
Gemini client ready (native API, x-goog-api-key, thinking enabled).


In [ ]:

# ── Load data ─────────────────────────────────────────────────────────────────
print("\nLoading NL profiles …", flush=True)
df_nl = pd.read_pickle(NL_PROFILE)
df_nl["pidp"] = pd.to_numeric(df_nl["pidp"], errors="coerce").astype("int64")
print(f"  {len(df_nl):,} respondents loaded.", flush=True)

avail_profile_cols = [col for col, _ in _PROFILE_COLS if col in df_nl.columns]
missing_cols = [col for col, _ in _PROFILE_COLS if col not in df_nl.columns]
if missing_cols:
    print(f"  Warning: missing profile columns (will encode as ?): {missing_cols}", flush=True)
print(f"  Profile columns available: {len(avail_profile_cols)}/{len(_PROFILE_COLS)}", flush=True)

print("Loading LA counts …", flush=True)
df_la = pd.read_csv(LA_COUNTS, dtype={"pidp": "int64", "ladcd": str, "ladnm": str, "n": "int64"})
print(f"  {len(df_la):,} rows, {df_la['ladcd'].nunique():,} LAs loaded.", flush=True)

if TEST_MODE and TEST_LA_CODES:
    df_la = df_la[df_la["ladcd"].isin(TEST_LA_CODES)].copy()
    print(f"  TEST MODE — {df_la['ladcd'].nunique()} LA(s): "
          f"{sorted(df_la['ladnm'].unique().tolist())}", flush=True)

keep_cols = ["pidp"] + avail_profile_cols
if hier_col:
    if hier_col not in df_nl.columns:
        raise ValueError(f"hier_col '{hier_col}' not found in NL profile table")
    if hier_col not in keep_cols:
        keep_cols.append(hier_col)

df_merged = df_la.merge(df_nl[keep_cols], on="pidp", how="inner")
if hier_col:
    df_merged.rename(columns={hier_col: "group"}, inplace=True)
print(f"  Merged: {len(df_merged):,} rows", flush=True)

print("  Building compact profile encodings …", flush=True)
df_merged["compact_profile"] = df_merged[avail_profile_cols].apply(
    lambda row: _encode_profile(row, avail_profile_cols), axis=1
)
print(f"  Unique compact profiles: {df_merged['compact_profile'].nunique():,}", flush=True)

# ── Build all LA × group combinations ────────────────────────────────────────
group_by = ["ladcd", "ladnm", "group"] if hier_col else ["ladcd", "ladnm"]
all_groups = (
    df_merged[group_by]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"\n{len(all_groups)} LA×group combination(s) to process:")
print(all_groups.to_string(index=False))

# ── Pre-compute proportional cluster allocations per LA ───────────────────────
# Each group's cluster count is scaled by its share of the LA's total population,
# with a budget of N_CLUSTERS_LOCAL × n_groups (so the average stays at N_CLUSTERS_LOCAL).
if hier_col:
    la_group_pop = (
        df_merged.groupby(["ladcd", "group"], sort=False)["n"]
        .sum()
        .reset_index()
        .rename(columns={"n": "pop"})
    )
    la_k_map: dict = {}   # (ladcd, group) -> k
    for ladcd, grp in la_group_pop.groupby("ladcd"):
        pop_dict  = dict(zip(grp["group"], grp["pop"]))
        budget    = N_CLUSTERS_LOCAL * len(pop_dict)
        allocs    = _lp.allocate_clusters(pop_dict, budget, min_per_group=1)
        for g, k_alloc in allocs.items():
            la_k_map[(ladcd, g)] = k_alloc
    print("\nCluster allocations per LA × group:")
    for (ladcd, g), k_alloc in sorted(la_k_map.items()):
        label = _lp.resolve_emp_label(g) or g
        print(f"  {ladcd}  {label:<30} → {k_alloc} clusters")
else:
    la_k_map = {}

# ── Skip already-done LA×group combos (resume support) ───────────────────────
# Track (ladcd, group) pairs so a partial run (some groups done, others not)
# resumes correctly rather than skipping the whole LA.
if API_OUT.exists():
    done_df = pd.read_csv(API_OUT, usecols=["ladcd"] + (["group"] if hier_col else []))
    if hier_col:
        done_pairs = set(zip(done_df["ladcd"].astype(str), done_df["group"].astype(str)))
        mask_done = all_groups.apply(
            lambda r: (str(r["ladcd"]), str(r["group"])) in done_pairs, axis=1
        )
    else:
        done_ladcds = set(done_df["ladcd"].unique())
        mask_done = all_groups["ladcd"].isin(done_ladcds)
    n_done = mask_done.sum()
    print(f"\n  Resuming — {n_done} LA×group(s) already done.")
else:
    mask_done = pd.Series(False, index=all_groups.index)

to_run = all_groups[~mask_done].copy()
n_done   = len(all_groups) - len(to_run)
n_total  = len(all_groups)
n_remain = len(to_run)
print(f"\n{'='*60}")
print(f"  Progress: {n_done} / {n_total} done  ({n_remain} remaining)")
if n_remain > 0 and n_done > 0:
    done_names = all_groups[mask_done]["ladnm"].unique()
    preview = ", ".join(done_names[:5].tolist())
    suffix = " …" if len(done_names) > 5 else ""
    print(f"  Already done: {preview}{suffix}")
print(f"{'='*60}")

# ── LLM call wrapper ──────────────────────────────────────────────────────────
MAX_MISSING_ASSIGNMENTS = 5   # tolerate up to this many missing tail assignments

def _call_llm(profiles_weights: list, k: int, context: str, emp_group=None) -> dict:
    system_msg = SYSTEM_PROMPT.format(k=k)
    prompt     = build_user_prompt(profiles_weights, k, context, emp_group=emp_group)
    print(f"    {len(profiles_weights)} unique profiles, ~{len(prompt)//4:,} tokens", flush=True)
    print(f"\n── SYSTEM PROMPT ──\n{system_msg}", flush=True)
    print(f"\n── USER PROMPT ──\n{prompt}\n──────────────────\n", flush=True)
    for attempt in range(3):
        try:
            t0 = time.time()
            content = _gemini_chat(system_msg, prompt, temperature=0.3)
            print(f"    {time.time()-t0:.1f}s", flush=True)
            parsed = json.loads(content)
            k_actual = len(parsed["clusters"])
            if k_actual < 1 or k_actual > k:
                raise ValueError(f"Expected 1-{k} clusters, got {k_actual}")
            if len(parsed.get("descriptions", [])) != k_actual:
                raise ValueError(f"Expected {k_actual} descriptions to match clusters, got {len(parsed.get('descriptions', []))}")
            n_assigned = len(parsed["assignments"])
            n_expected = len(profiles_weights)
            if n_assigned > n_expected or (n_expected - n_assigned) > MAX_MISSING_ASSIGNMENTS:
                raise ValueError(f"Expected {n_expected} assignments, got {n_assigned}")
            if n_assigned < n_expected:
                print(f"    Warning: {n_expected - n_assigned} missing assignment(s) — padding with 0", flush=True)
                parsed["assignments"] += [0] * (n_expected - n_assigned)
            parsed["clusters"]     = [sanitise_name(n) for n in parsed["clusters"]]
            parsed["descriptions"] = [sanitise_name(d) for d in parsed["descriptions"]]
            for name, desc in zip(parsed["clusters"], parsed["descriptions"]):
                print(f"    • {name}: {desc}", flush=True)
            return parsed
        except Exception as exc:
            print(f"    attempt {attempt+1} failed: {exc}", flush=True)
            if attempt < 2:
                time.sleep(RETRY_DELAY)
    raise RuntimeError("LLM call failed after 3 attempts.")

# ── Run LLM per LA × group, saving each LA immediately ───────────────────────
df_features = df_nl.drop(columns=["nl_profile"], errors="ignore")
leading = ["ladcd", "ladnm"] + (["group"] if hier_col else [])

print(f"\nStarting LLM calls for {len(to_run)} LA×group(s) …", flush=True)

for _, meta in to_run.iterrows():
    ladcd_val = meta["ladcd"]
    ladnm_val = meta["ladnm"]
    group_val = meta.get("group")

    emp_label = resolve_emp_label(group_val) if group_val is not None else None
    context   = f"{ladnm_val} — {emp_label or group_val or 'all respondents'}"

    mask = df_merged["ladcd"] == ladcd_val
    if group_val is not None:
        mask &= df_merged["group"] == group_val
    slice_df = df_merged[mask].copy()

    pidp_weights = (
        slice_df.groupby("pidp", sort=False)
        .agg(n=("n", "sum"), compact_profile=("compact_profile", "first"))
        .reset_index()
        .sort_values("n", ascending=False)
        .reset_index(drop=True)
    )

    top = pidp_weights.iloc[:MAX_PIDPS_PER_GROUP].copy()

    deduped = (
        top.groupby("compact_profile", sort=False)
        .agg(n=("n", "sum"))
        .reset_index()
        .sort_values("n", ascending=False)
        .reset_index(drop=True)
    )

    # Proportional allocation (hierarchical) or fixed (flat)
    if hier_col and group_val is not None:
        k_target = la_k_map.get((ladcd_val, group_val), N_CLUSTERS_LOCAL)
    else:
        k_target = N_CLUSTERS_LOCAL
    k = min(k_target, len(deduped))

    print(f"\n[{ladnm_val} / {emp_label or group_val or '—'}]  "
          f"{len(top)} pidps → {len(deduped)} unique profiles → {k} clusters …",
          flush=True)

    result        = _call_llm(
        list(zip(deduped["compact_profile"], deduped["n"].astype(int))),
        k, context,
        emp_group=group_val,
    )
    cluster_names = result["clusters"]
    cluster_descs = result["descriptions"]

    profile_to_cluster = {
        row["compact_profile"]: min(int(result["assignments"][i]), len(cluster_names) - 1)
        for i, row in deduped.iterrows()
    }

    la_pidp_rows = []
    for _, row in top.iterrows():
        ci = profile_to_cluster.get(row["compact_profile"], 0)
        la_pidp_rows.append({
            "ladcd":             ladcd_val,
            "ladnm":             ladnm_val,
            "group":             group_val,
            "pidp":              int(row["pidp"]),
            "n_sipher_rows":     int(row["n"]),
            "cluster":           ci + 1,
            "tribe_label":       cluster_names[ci],
            "tribe_description": cluster_descs[ci],
        })

    la_df_pidp = pd.DataFrame(la_pidp_rows)
    la_df_full = la_df_pidp.merge(df_features, on="pidp", how="left")
    la_df_full["ladnm"] = ladnm_val
    if hier_col:
        la_df_full["group"] = group_val

    la_summary = _cs.make_cluster_summary(la_df_full, "cluster", wave=WAVE)

    name_map = la_df_pidp.drop_duplicates("cluster").set_index("cluster")["tribe_label"].to_dict()
    desc_map = la_df_pidp.drop_duplicates("cluster").set_index("cluster")["tribe_description"].to_dict()

    la_summary["tribe_label"]       = la_summary["cluster_id"].map(name_map).fillna(la_summary["tribe_label"])
    la_summary["tribe_description"] = la_summary["cluster_id"].map(desc_map)
    la_summary["reasoning"] = result.get("reasoning", "")
    la_summary["ladcd"] = ladcd_val
    la_summary["ladnm"] = ladnm_val
    if hier_col:
        la_summary["group"] = group_val

    la_summary = la_summary[leading + [c for c in la_summary.columns if c not in leading]]

    write_header = not API_OUT.exists()
    la_summary.to_csv(API_OUT, mode="a", header=write_header, index=False)
    print(f"  Saved {len(la_summary)} rows → {API_OUT}", flush=True)

# ── Load full output and print preview ───────────────────────────────────────
df_summary = pd.read_csv(API_OUT)
print(f"\nDone. {len(df_summary)} total cluster rows in {API_OUT}")

preview_cols = leading + ["cluster_id", "tribe_label", "tribe_description", "n_respondents", "size"]
print(df_summary[[c for c in preview_cols if c in df_summary.columns]].to_string(index=False))



Loading NL profiles …
  27,330 respondents loaded.
  Profile columns available: 9/9
Loading LA counts …
  7,969,122 rows, 346 LAs loaded.
  TEST MODE — 4 LA(s): ['Hounslow', 'Islington', 'Newham', 'Tower Hamlets']
  Merged: 95,268 rows
  Building compact profile encodings …
  Unique compact profiles: 22,165

24 LA×group combination(s) to process:
    ladcd         ladnm  group
E09000018      Hounslow    5.0
E09000019     Islington    5.0
E09000025        Newham    5.0
E09000030 Tower Hamlets    5.0
E09000018      Hounslow    1.0
E09000019     Islington    1.0
E09000025        Newham    1.0
E09000030 Tower Hamlets    1.0
E09000018      Hounslow    8.0
E09000019     Islington    8.0
E09000025        Newham    8.0
E09000030 Tower Hamlets    8.0
E09000018      Hounslow    7.0
E09000019     Islington    7.0
E09000025        Newham    7.0
E09000030 Tower Hamlets    7.0
E09000018      Hounslow    4.0
E09000019     Islington    4.0
E09000025        Newham    4.0
E09000030 Tower Hamlets    4.0